In [0]:
-- sql/08_papel_narrativo_v0.sql
-- Papéis narrativos declarados para as contas mais mencionadas de cada caso (P4 — contágio e legitimação institucional).
-- Camada interpretativa sobre a Gold: rótulos DECLARADOS por leitura do caso, não medidos pelo pipeline.
-- Só conta_id (surrogate da Silver); nenhum nome, handle ou texto. Reversível linha a linha.
-- Semente da gold.rotulo_publico (v2), que acrescentará nomes públicos conta a conta.
-- Decidido em 08/09/2026 (Bloco 4). Versão v0 = 40 contas (top-25 por menções recebidas, exclusões documentadas).

USE CATALOG scapegoat;

CREATE TABLE IF NOT EXISTS gold.papel_narrativo_v0 (
  caso           STRING  NOT NULL,
  conta_id       BIGINT  NOT NULL,
  papel_inicial  STRING  NOT NULL,
  papel_final    STRING,
  dia_virada     INT,
  origem         STRING  NOT NULL,
  justificativa  STRING,
  decidido_em    DATE    NOT NULL,
  CONSTRAINT pk_papel_narrativo_v0 PRIMARY KEY (caso, conta_id) NOT ENFORCED
)
COMMENT 'Papel narrativo (girardiano) de contas selecionadas por caso: instituicao_legitimadora | veiculo | lider_acusacao | acusador | aliado_do_alvo | aliado_afastado | vitima_secundaria | outro. Rótulos DECLARADOS por leitura do caso (origem = declarado), não derivados do pipeline; papel_final/dia_virada registram mudança de lado (dia em dias_desde_estopim). Só conta_id, sem nome; semente da rotulo_publico (v2). Grão: 1 linha por caso × conta.';

-- CHECK constraints: no Delta entram depois do CREATE (só PK/FK são aceitas inline). São impostas na escrita.
ALTER TABLE gold.papel_narrativo_v0 ADD CONSTRAINT ck_papel_inicial CHECK (papel_inicial IN ('instituicao_legitimadora','veiculo','lider_acusacao','acusador','aliado_do_alvo','aliado_afastado','vitima_secundaria','outro'));
ALTER TABLE gold.papel_narrativo_v0 ADD CONSTRAINT ck_papel_final   CHECK (papel_final IS NULL OR papel_final IN ('instituicao_legitimadora','veiculo','lider_acusacao','acusador','aliado_do_alvo','aliado_afastado','vitima_secundaria','outro'));
ALTER TABLE gold.papel_narrativo_v0 ADD CONSTRAINT ck_virada        CHECK ((papel_final IS NULL AND dia_virada IS NULL) OR (papel_final IS NOT NULL AND dia_virada IS NOT NULL));
ALTER TABLE gold.papel_narrativo_v0 ADD CONSTRAINT ck_origem        CHECK (origem IN ('declarado','medido'));

ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN caso          COMMENT 'Slug do caso (monark | arthur_do_val). Chave para dim_conta_papel.';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN conta_id      COMMENT 'Chave surrogate de silver.conta. Sem handle na Gold.';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN papel_inicial COMMENT 'Papel narrativo no início do episódio (ou único, se não há virada).';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN papel_final   COMMENT 'Papel após a virada; NULL se o papel não muda.';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN dia_virada    COMMENT 'Dia da virada em dias_desde_estopim (0 = estopim); NULL se não há virada. Declarado, não medido.';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN origem        COMMENT 'declarado = atribuído por leitura do caso; medido = derivado de métrica (nenhum em v0).';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN justificativa COMMENT 'Motivo do rótulo, sem nome nem texto de postagem.';
ALTER TABLE gold.papel_narrativo_v0 ALTER COLUMN decidido_em   COMMENT 'Data da decisão.';

ALTER TABLE gold.papel_narrativo_v0 SET TAGS ('layer' = 'gold', 'owner' = 'carlos', 'domain' = 'scapegoating',
                                              'refresh' = 'manual', 'classification' = 'pseudonimizado', 'origem' = 'declarado');

-- Carga v0. Primeira carga em tabela vazia; recarga = INSERT OVERWRITE (nunca DELETE).
INSERT OVERWRITE gold.papel_narrativo_v0 VALUES
-- ===== monark =====
('monark', 4429, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada pela multidão, rompeu o patrocínio', DATE '2026-09-08'),
('monark', 5193, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 5482, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'o programa; pressionado, afastou o alvo', DATE '2026-09-08'),
('monark', 4988, 'vitima_secundaria',        NULL, NULL, 'declarado', 'segundo alvo da mesma onda, por contágio temático (gesto em outro programa); demitido', DATE '2026-09-08'),
('monark', 5484, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 4596, 'vitima_secundaria',        NULL, NULL, 'declarado', 'convidado do episódio, relativizou a mesma tese e foi atacado por cumplicidade', DATE '2026-09-08'),
('monark', 5388, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 4655, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 4241, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 5100, 'veiculo',                  NULL, NULL, 'declarado', 'plataforma de vídeo; mencionada por links', DATE '2026-09-08'),
('monark', 2611, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'emissora empregadora do segundo alvo; pressionada, demitiu', DATE '2026-09-08'),
('monark', 4336, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
('monark', 4834, 'aliado_do_alvo', 'acusador', 1, 'declarado', 'sócio e co-apresentador; aliado até o afastamento do alvo (09/02, dia 1), depois do lado da acusação', DATE '2026-09-08'),
('monark', 3086, 'veiculo',                  NULL, NULL, 'declarado', 'agregador de notícias', DATE '2026-09-08'),
('monark', 4992, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'instituição de memória que se manifestou publicamente contra o alvo', DATE '2026-09-08'),
('monark', 4295, 'lider_acusacao',           NULL, NULL, 'declarado', 'coletivo que pressionou os patrocinadores; maior parcela acusadora do caso', DATE '2026-09-08'),
('monark', 5485, 'veiculo',                  NULL, NULL, 'declarado', 'veículo de notícias', DATE '2026-09-08'),
('monark', 5436, 'veiculo',                  NULL, NULL, 'declarado', 'perfil de mobilização/agregação', DATE '2026-09-08'),
('monark', 1065, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'patrocinadora do programa; pressionada, rompeu', DATE '2026-09-08'),
-- ===== arthur_do_val =====
('arthur_do_val', 4392,  'aliado_do_alvo', 'aliado_afastado', 4, 'declarado', 'movimento do alvo; distanciou-se em 08/03 (dia 4) sem acusar; vínculo retomado após a janela de coleta', DATE '2026-09-08'),
('arthur_do_val', 3770,  'aliado_do_alvo', 'aliado_afastado', 4, 'declarado', 'dirigente do movimento; companheiro na viagem da polêmica anterior; acompanha o distanciamento do movimento (dia 4)', DATE '2026-09-08'),
('arthur_do_val', 4326,  'outro',                    NULL, NULL, 'declarado', 'antagonista da polêmica anterior (pré-crise); não pertence a este episódio', DATE '2026-09-08'),
('arthur_do_val', 15546, 'outro',                    NULL, NULL, 'declarado', 'antagonista da polêmica anterior (pré-crise)', DATE '2026-09-08'),
('arthur_do_val', 5420,  'outro',                    NULL, NULL, 'declarado', 'antagonista da polêmica anterior (pré-crise)', DATE '2026-09-08'),
('arthur_do_val', 4939,  'aliado_do_alvo',           NULL, NULL, 'declarado', 'candidato apoiado pelo movimento do alvo; cobrado a se posicionar', DATE '2026-09-08'),
('arthur_do_val', 11579, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'parlamentar estadual marcado em bloco no dia 8 para cobrar a cassação', DATE '2026-09-08'),
('arthur_do_val', 5100,  'veiculo',                  NULL, NULL, 'declarado', 'plataforma de vídeo; mencionada por links', DATE '2026-09-08'),
('arthur_do_val', 8610,  'instituicao_legitimadora', NULL, NULL, 'declarado', 'parlamentar estadual marcado em bloco no dia 8', DATE '2026-09-08'),
('arthur_do_val', 1808,  'instituicao_legitimadora', NULL, NULL, 'declarado', 'parlamentar estadual marcado em bloco no dia 8', DATE '2026-09-08'),
('arthur_do_val', 16274, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'parlamentar estadual marcado em bloco no dia 8; entra no caso nesse dia', DATE '2026-09-08'),
('arthur_do_val', 4596,  'aliado_do_alvo', 'aliado_afastado', 4, 'declarado', 'parlamentar do movimento do alvo; acompanha o distanciamento (dia 4)', DATE '2026-09-08'),
('arthur_do_val', 4687,  'aliado_do_alvo', 'aliado_afastado', 4, 'declarado', 'comentarista do movimento do alvo; acompanha o distanciamento (dia 4)', DATE '2026-09-08'),
('arthur_do_val', 13464, 'lider_acusacao',           NULL, NULL, 'declarado', 'autora com muitas postagens acusadoras (69), respondida por poucos', DATE '2026-09-08'),
('arthur_do_val', 5203,  'outro',                    NULL, NULL, 'declarado', 'figura da polêmica anterior (pré-crise)', DATE '2026-09-08'),
('arthur_do_val', 15172, 'instituicao_legitimadora', NULL, NULL, 'declarado', 'casa legislativa responsável pelo processo de cassação', DATE '2026-09-08'),
('arthur_do_val', 15545, 'lider_acusacao',           NULL, NULL, 'declarado', 'parlamentar que liderou a representação pela cassação', DATE '2026-09-08'),
('arthur_do_val', 1075,  'veiculo',                  NULL, NULL, 'declarado', 'veículo de notícias', DATE '2026-09-08'),
('arthur_do_val', 14997, 'outro',                    NULL, NULL, 'declarado', 'figura da polêmica anterior (pré-crise)', DATE '2026-09-08'),
('arthur_do_val', 668,   'veiculo',                  NULL, NULL, 'declarado', 'veículo de notícias', DATE '2026-09-08'),
('arthur_do_val', 14729, 'outro',                    NULL, NULL, 'declarado', 'figura da polêmica anterior (pré-crise)', DATE '2026-09-08');

-- Exclusões documentadas (posições do top-25 que NÃO entram): monark 12 (comentarista, fora da classificação), 15/16/17 (2–4 mencionadores: ruído),
-- 10 (convidada; menção de contexto, não confirmada como vítima), 25 (comentarista; fora); arthur 11/20/22 (poucos mencionadores: ruído), 23 (aliado, deixado de lado).

-- Conferências
SELECT caso, COUNT(*) AS linhas FROM gold.papel_narrativo_v0 GROUP BY caso;            -- esperado: monark 19, arthur_do_val 21
SELECT p.caso, p.conta_id FROM gold.papel_narrativo_v0 p
LEFT JOIN gold.dim_conta_papel d ON d.caso = p.caso AND d.conta_id = p.conta_id
WHERE d.conta_id IS NULL OR d.papel_principal = 'alvo';                                   -- esperado: 0 linhas
SELECT caso, papel_inicial, COUNT(*) AS n FROM gold.papel_narrativo_v0 GROUP BY caso, papel_inicial ORDER BY caso, n DESC;
